# 20. 검증셋 샘플링 및 통합 저장 (Sampled Validation Set)

이 노트북은 학습 효율과 성능 지표의 안정성을 위해 **검증셋을 전략적으로 샘플링(Sampling)**하고, 이를 **학습용 트레인 서브셋과 동일한 경로에 통합 관리**하도록 구성합니다.

**핵심 전략:**
1. **Sampled Validation**: 모든 고장 샘플 + 가중치 샘플링된 정상 데이터를 묶어 '고밀도 검증셋'을 만듭니다.
2. **통합 관리**: 생성된 검증셋을 각 시드별 학습 데이터 폴더(`data/train_subsets/seed_{SEED}/`)에 함께 저장하여 모델 학습 세트를 완성합니다.
3. **재현성**: 특정 시드(`TARGET_SEED`)를 사용하여 샘플링 구성을 고정합니다.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
from pathlib import Path
from src.train_core import AsymmetricSampler
import config.train_config as cfg

# ── [독립 설정 구역] ────────────────────────────────────────
TARGET_SEED = 42      # 샘플링 시드 번호
NEG_RATIO   = 100.0   # 정상:고장 비율 (기본 1:100)

# [저장 정보]
# 학습용 서브셋과 동일한 폴더에 'val_sampled.parquet' 이름으로 저장됩니다.
SAVE_DIR = Path("..") / "data" / "train_subsets" / f"seed_{TARGET_SEED}"
SAVE_PATH = SAVE_DIR / "val_sampled.parquet"

print(f"🚀 검증셋 샘플링 준비: Seed={TARGET_SEED}, Ratio=1:{NEG_RATIO}")
print(f"📂 통합 저장 위치: {SAVE_PATH.resolve()}")

🖥️  LightGBM GPU 가용성: [OK] 사용 가능
🚀 검증셋 샘플링 준비: Seed=42, Ratio=1:100.0
📂 통합 저장 위치: C:\Workspace\COIN\ML_HDD\data\train_subsets\seed_42\val_sampled.parquet


## 1. 원본 검증 데이터 로드

In [2]:
val_path = Path(cfg.VAL_TUNE_PATH)
print(f"📂 원본 데이터 로딩 중... ({val_path})")
df_val = pd.read_parquet(val_path)
print(f"✅ 로드 완료: {len(df_val):,} rows")

📂 원본 데이터 로딩 중... (c:\Workspace\COIN\ML_HDD\data\split_group_stratified\val_tune.parquet)
✅ 로드 완료: 7,902,193 rows


## 2. 전략적 샘플링 실행 (Asymmetric Sampling)
고장 전수와 Near-failure 가중치가 적용된 정상 데이터를 추출합니다.

In [3]:
sampler = AsymmetricSampler(
    n_subsets=1,
    neg_ratio=NEG_RATIO,
    near_window=cfg.SAMPLER_KWARGS['near_window'],
    near_weight=cfg.SAMPLER_KWARGS['near_weight'],
    seed=TARGET_SEED
)

print(f"🔧 샘플링 중... (비율 1:{NEG_RATIO}, SEED: {TARGET_SEED})")
sampled_subsets = sampler.split(df_val, target_col=cfg.TARGET_COL)
df_sampled_val = sampled_subsets[0]

pos_count = int(df_sampled_val[cfg.TARGET_COL].sum())
neg_count = len(df_sampled_val) - pos_count

print(f"\n✅ 샘플링 결과:")
print(f"   - 전체 rows: {len(df_sampled_val):,}")
print(f"   - 고장(Pos): {pos_count:,} (전수 포함)")
print(f"   - 정상(Neg): {neg_count:,}")
print(f"   - 불균형 비율: 1:{neg_count/pos_count:.1f}")

🔧 샘플링 중... (비율 1:100.0, SEED: 42)

✅ 샘플링 결과:
   - 전체 rows: 559,136
   - 고장(Pos): 5,536 (전수 포함)
   - 정상(Neg): 553,600
   - 불균형 비율: 1:100.0


## 3. 학습 세트 폴더에 통합 저장

In [4]:
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# 학습용 데이터(subset_*.parquet)와 함께 관리됩니다.
df_sampled_val.to_parquet(SAVE_PATH, index=False)

print(f"✨ 샘플링된 검증셋 저장 완료!")
print(f"📍 경로: {SAVE_PATH.resolve()}")
print(f"🔗 이 데이터는 이제 '{SAVE_DIR.name}' 세트의 일부로 학습 시 사용됩니다.")

✨ 샘플링된 검증셋 저장 완료!
📍 경로: C:\Workspace\COIN\ML_HDD\data\train_subsets\seed_42\val_sampled.parquet
🔗 이 데이터는 이제 'seed_42' 세트의 일부로 학습 시 사용됩니다.
